In [1]:
# Imports
import numpy as np
import pandas as pd 
from scipy.stats import linregress
from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
def calculate_elo_rate(teams, data, initial_rating=2000, k=140, width=None, alpha=None, weights=False, lowerlim=float("-inf")):
    # Dictionary to keep track of current ratings for each team
    team_dict = {}
    for team in teams:
        team_dict[team] = initial_rating
    if not width:
        width = initial_rating
    # Lists to store ratings for each team in each game
    r1, r2 = [], []
    loss = []
    margin_of_victory = 1
    # Iterate through the game data
    for wteam, lteam, ws, ls, w  in tqdm(zip(data.WTeamID, data.LTeamID, data.WScore, data.LScore, data.weight), total=len(data)):
        # Calculate expected outcomes based on Elo ratings
        rateW = 1 / (1 + 10 ** ((team_dict[lteam] - team_dict[wteam]) / width))
        rateL = 1 / (1 + 10 ** ((team_dict[wteam] - team_dict[lteam]) / width))
        if alpha:
            margin_of_victory = (ws - ls)/alpha
        # Update ratings for winning and losing teams
        team_dict[wteam] += w * k * margin_of_victory * (1 - rateW)
        team_dict[lteam] += w * k * margin_of_victory * (0 - rateL)
        # Ensure that ratings do not go below lower limit
        if team_dict[lteam] < lowerlim:
            team_dict[lteam] = lowerlim
        # Append current ratings for teams to lists
        r1.append(team_dict[wteam])
        r2.append(team_dict[lteam])
        loss.append((1-rateW)**2)
    return r1, r2, loss

def create_elo_data(teams, data, initial_rating=2000, k=140, width=None, alpha=None, weights=None, lowerlim=float("-inf")):
    if isinstance(weights, (list, np.ndarray, pd.Series)):
        data['weight'] = weights
    else:
        data['weight'] = 1
    r1, r2, loss = calculate_elo_rate(teams, data, initial_rating, k, width, alpha, weights, lowerlim)
    # Calculate loss only on tourney results
    loss = np.mean(np.array(loss)[data.tourney == 1])
    print(f"=== Brier Score: {loss:.5f} (Only  Tournaments) ===")
    # Concatenate arrays vertically
    seasons = np.concatenate([data.Season, data.Season])
    days = np.concatenate([data.DayNum, data.DayNum])
    teams = np.concatenate([data.WTeamID, data.LTeamID])
    tourney = np.concatenate([data.tourney, data.tourney])
    ratings = np.concatenate([r1, r2])
    # Create a DataFrame
    rating_df = pd.DataFrame({
        'Season': seasons,
        'DayNum': days,
        'TeamID': teams,
        'Rating': ratings,
        'Tourney': tourney
    })
    # Sort DataFrame and remove tournament data
    rating_df.sort_values(['TeamID', 'Season', 'DayNum'], inplace=True)
    rating_df = rating_df[rating_df['Tourney'] == 0]
    grouped = rating_df.groupby(['TeamID', 'Season'])
    results = grouped['Rating'].agg(['mean', 'median', 'std', 'min', 'max', 'last'])
    results.columns = ['Rating_Mean', 'Rating_Median', 'Rating_Std', 'Rating_Min', 'Rating_Max', 'Rating_Last']
    results['Rating_Trend'] = grouped.apply(lambda x: linregress(range(len(x)), x['Rating']).slope, include_groups=False)
    results.reset_index(inplace=True)
    return results, loss

In [3]:
# Load and Process Data Men's Tourney
regular_m = pd.read_csv('/kaggle/input/competitions/march-machine-learning-mania-2026/MRegularSeasonCompactResults.csv')
tourney_m = pd.read_csv('/kaggle/input/competitions/march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv')
teams_m = pd.read_csv('/kaggle/input/competitions/march-machine-learning-mania-2026/MTeams.csv')
regular_m['tourney'] = 0
tourney_m['tourney'] = 1
regular_m['weight'] = 1
tourney_m['weight'] = 0.75
data_m = pd.concat([regular_m, tourney_m])
data_m.sort_values(['Season', 'DayNum'], inplace=True)
data_m.reset_index(inplace=True, drop=True)
initial_rating = width = 1200 
elo_df_men, _ = create_elo_data(teams_m.TeamID, data_m, initial_rating=initial_rating, k=125, width=width, alpha=None, weights=data_m['weight'])
# Load and Process Data Women's Tourney
regular_w = pd.read_csv('/kaggle/input/competitions/march-machine-learning-mania-2026/WRegularSeasonCompactResults.csv')
tourney_w = pd.read_csv('/kaggle/input/competitions/march-machine-learning-mania-2026/WNCAATourneyCompactResults.csv')
teams_w = pd.read_csv('/kaggle/input/competitions/march-machine-learning-mania-2026/WTeams.csv')
regular_w['tourney'] = 0
tourney_w['tourney'] = 1
regular_w['weight'] = 0.95
tourney_w['weight'] = 1
data_w = pd.concat([regular_w, tourney_w])
data_w.sort_values(['Season', 'DayNum'], inplace=True)
data_w.reset_index(inplace=True, drop=True)
elo_df_women, _ = create_elo_data(teams_w.TeamID, data_w, initial_rating=initial_rating, k=190, width=width, alpha=None, weights=data_w['weight'])


100%|██████████| 201162/201162 [00:00<00:00, 374243.99it/s]


=== Brier Score: 0.18700 (Only  Tournaments) ===


100%|██████████| 144224/144224 [00:00<00:00, 406672.54it/s]


=== Brier Score: 0.14709 (Only  Tournaments) ===


In [4]:
# Prepare direct submission
submission = pd.read_csv("/kaggle/input/competitions/march-machine-learning-mania-2026/SampleSubmissionStage2.csv")
# Split the ID into Season, T1_TeamID, and T2_TeamID
sub = submission.ID.str.split('_', expand=True).astype(int)
sub.columns = ["Season", "T1_TeamID", "T2_TeamID"]
# Turn elo dfs into dict
elo_dict = pd.concat([elo_df_women[elo_df_women.Season == 2026],elo_df_men[elo_df_men.Season == 2026]]).set_index("TeamID")["Rating_Last"].to_dict()
# Calculate probabilities
submission.Pred = 1 / (1 + 10**((sub.T2_TeamID.map(elo_dict) - sub.T1_TeamID.map(elo_dict))/width))
submission.to_csv("submission.csv", index=False)
submission.head()

,ID,Pred
0,2026_1101_1102,0.737462
1,2026_1101_1103,0.070974
2,2026_1101_1104,0.014726
3,2026_1101_1105,0.766540
4,2026_1101_1106,0.820310


ref : https://www.kaggle.com/code/lennarthaupts/calculate-elo-ratings